In [89]:
#analysis 
import pandas as pd
import numpy as np

#visualization
import seaborn as sns
import matplotlib.pyplot as plt

#training models
from sklearn.model_selection import train_test_split, GridSearchCV

#scaling and preprocessing
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder
from sklearn.compose import ColumnTransformer

#Modeling
from sklearn.pipeline import Pipeline
from sklearn.ensemble import (
    RandomForestClassifier, 
    AdaBoostClassifier, 
    GradientBoostingClassifier
)
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from xgboost import XGBClassifier


#metrics
from sklearn.metrics import (
    classification_report, 
    roc_auc_score, 
    accuracy_score, 
    precision_score, 
    recall_score, 
    f1_score
)
#saving and loading models
import joblib
import os



sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = [10, 6]
plt.rcParams['font.size'] = 12

In [90]:
df =pd.read_csv('processed_churn_data.csv')
df

,ord__Contract,ord__Tenure_Group,nom__PhoneService_Yes,nom__PaperlessBilling_Yes,nom__PaymentMethod_Credit card (automatic),nom__PaymentMethod_Electronic check,nom__PaymentMethod_Mailed check,remainder__customerID,remainder__tenure,remainder__MonthlyCharges,remainder__TotalCharges,remainder__Churn
0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,7590-VHVEG,1,29.85,29.85,No
1,1.0,2.0,1.0,0.0,0.0,0.0,1.0,5575-GNVDE,34,56.95,1889.5,No
2,0.0,0.0,1.0,1.0,0.0,0.0,1.0,3668-QPYBK,2,53.85,108.15,Yes
3,1.0,2.0,0.0,0.0,0.0,0.0,0.0,7795-CFOCW,45,42.30,1840.75,No
4,0.0,0.0,1.0,1.0,0.0,1.0,0.0,9237-HQITU,2,70.70,151.65,Yes
...,...,...,...,...,...,...,...,...,...,...,...,...
7037,2.0,2.0,1.0,1.0,0.0,0.0,0.0,2569-WGERO,72,21.15,1419.4,No
7038,1.0,1.0,1.0,1.0,0.0,0.0,1.0,6840-RESVB,24,84.80,1990.5,No
7039,1.0,2.0,1.0,1.0,1.0,0.0,0.0,2234-XADUH,72,103.20,7362.9,No
7040,0.0,1.0,0.0,1.0,0.0,1.0,0.0,4801-JZAZL,11,29.60,346.45,No


In [91]:
# Convert the column to numeric, forcing errors (like ' ') into NaN, then filling them with 0
df['remainder__TotalCharges'] = pd.to_numeric(df['remainder__TotalCharges'], errors='coerce').fillna(0)

In [92]:
y = df['remainder__Churn'].map({'No': 0, 'Yes': 1})
X = df.drop(columns=['remainder__customerID', 'remainder__Churn'], errors='ignore')

In [93]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42,stratify=y)

In [94]:
scaler = StandardScaler()
x_train_scaled = scaler.fit_transform(x_train)
x_test_scaled = scaler.transform(x_test)

print("Training set shape:", x_train_scaled.shape)
print("Test set shape:", x_test_scaled.shape)

Training set shape: (5633, 10)
Test set shape: (1409, 10)


In [100]:
models = {
    'Logistic Regression': LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42),
    'Random Forest': RandomForestClassifier(class_weight='balanced', random_state=42),
    'XGBoost': XGBClassifier(n_estimators=200, max_depth=4, learning_rate=0.1, random_state=42),
    'AdaBoost': AdaBoostClassifier(random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(random_state=42),
    'KNN': KNeighborsClassifier(),
    'SVM': SVC(probability=True, class_weight='balanced', random_state=42), # probability=True is required for predict_proba
    'Naive Bayes': GaussianNB()
}

In [101]:
param_grids = {
    'Logistic Regression': {
        'C': [0.01, 0.1, 1.0, 10.0], 
    },
    'Random Forest': {
        'n_estimators': [100, 200], 
        'max_depth': [4, 8, 12], 
        'min_samples_split': [2, 5]
    },
    'XGBoost': {
        'n_estimators': [100, 200], 
        'max_depth': [3, 4, 6], 
        'learning_rate': [0.01, 0.05, 0.1], 
    },
    'AdaBoost': {
        'n_estimators': [50, 100, 200], 
        'learning_rate': [0.01, 0.1, 1.0]
    },
    'Gradient Boosting': {
        'n_estimators': [100, 200], 
        'learning_rate': [0.01, 0.1, 0.2], 
        'max_depth': [3, 5]
    },
    'KNN': {
        'n_neighbors': [3, 5, 7, 9], 
    },
    'SVM': {
        'C': [0.1, 1.0, 10.0], 
        'kernel': ['linear', 'rbf']
    },
    'Naive Bayes': {

    }
}



In [102]:
for name, model in models.items():
    print(f"Running GridSearchCV for {name}...")
    
    grid = GridSearchCV(
        estimator=model,
        param_grid=param_grids[name],  # Pulls the correct grid for the current model
        scoring='roc_auc',
        cv=5,
        n_jobs=-1,
        verbose=1
    )

Running GridSearchCV for Logistic Regression...
Running GridSearchCV for Random Forest...
Running GridSearchCV for XGBoost...
Running GridSearchCV for AdaBoost...
Running GridSearchCV for Gradient Boosting...
Running GridSearchCV for KNN...
Running GridSearchCV for SVM...
Running GridSearchCV for Naive Bayes...


In [103]:
fitted_grids = {}
results = []

for name, model in models.items():
    print(f"--> Running GridSearchCV for {name}...")
    
    # We still optimize the grid search based on ROC-AUC
    grid = GridSearchCV(
        estimator=model,
        param_grid=param_grids[name],
        scoring='roc_auc',
        cv=5,
        n_jobs=-1,
        verbose=1
    )
    
    grid.fit(x_train_scaled, y_train)
    fitted_grids[name] = grid
    best_estimator = grid.best_estimator_
    
    # Generate predictions for the test set
    y_pred_proba = best_estimator.predict_proba(x_test_scaled)[:, 1]
    y_pred = best_estimator.predict(x_test_scaled)
    
    # Calculate comprehensive test metrics
    test_roc_auc = roc_auc_score(y_test, y_pred_proba)
    test_accuracy = accuracy_score(y_test, y_pred)
    test_precision = precision_score(y_test, y_pred)
    test_recall = recall_score(y_test, y_pred)
    test_f1 = f1_score(y_test, y_pred)
    
    results.append({
        'Model': name,
        'CV ROC-AUC': round(grid.best_score_, 4),
        'Test ROC-AUC': round(test_roc_auc, 4),
        'Test Accuracy': round(test_accuracy, 4),
        'Test Precision': round(test_precision, 4),
        'Test Recall': round(test_recall, 4),
        'Test F1-Score': round(test_f1, 4),
        'Best Parameters': grid.best_params_
    })

# Display the expanded comparison table
comparison_df = pd.DataFrame(results).sort_values(by='Test ROC-AUC', ascending=False)
display(comparison_df)

--> Running GridSearchCV for Logistic Regression...
Fitting 5 folds for each of 4 candidates, totalling 20 fits
--> Running GridSearchCV for Random Forest...
Fitting 5 folds for each of 12 candidates, totalling 60 fits
--> Running GridSearchCV for XGBoost...
Fitting 5 folds for each of 18 candidates, totalling 90 fits
--> Running GridSearchCV for AdaBoost...
Fitting 5 folds for each of 9 candidates, totalling 45 fits
--> Running GridSearchCV for Gradient Boosting...
Fitting 5 folds for each of 12 candidates, totalling 60 fits
--> Running GridSearchCV for KNN...
Fitting 5 folds for each of 4 candidates, totalling 20 fits
--> Running GridSearchCV for SVM...
Fitting 5 folds for each of 6 candidates, totalling 30 fits
--> Running GridSearchCV for Naive Bayes...
Fitting 5 folds for each of 1 candidates, totalling 5 fits


,Model,CV ROC-AUC,Test ROC-AUC,Test Accuracy,Test Precision,Test Recall,Test F1-Score,Best Parameters
1,Random Forest,0.8399,0.8446,0.7622,0.5369,0.7594,0.6290,"{'max_depth': 8, 'min_samples_split': 2, 'n_es..."
2,XGBoost,0.8405,0.8433,0.7977,0.6654,0.4786,0.5568,"{'learning_rate': 0.05, 'max_depth': 3, 'n_est..."
4,Gradient Boosting,0.8390,0.8399,0.7956,0.6581,0.4786,0.5542,"{'learning_rate': 0.1, 'max_depth': 3, 'n_esti..."
0,Logistic Regression,0.8402,0.8373,0.7395,0.5059,0.7968,0.6189,{'C': 0.1}
6,SVM,0.8396,0.8368,0.7289,0.4937,0.8422,0.6225,"{'C': 0.1, 'kernel': 'linear'}"
3,AdaBoost,0.8397,0.8358,0.7906,0.6426,0.4759,0.5469,"{'learning_rate': 1.0, 'n_estimators': 200}"
7,Naive Bayes,0.8179,0.8162,0.7466,0.5163,0.7219,0.6020,{}
5,KNN,0.8075,0.8121,0.7821,0.6084,0.5027,0.5505,{'n_neighbors': 9}


In [106]:
best_model_name = comparison_df.iloc[0]['Model']
best_model = fitted_grids[best_model_name].best_estimator_

print(f"Top Performer: {best_model_name}")
print(f"Optimal Parameters: {fitted_grids[best_model_name].best_params_}")

Top Performer: Random Forest
Optimal Parameters: {'max_depth': 8, 'min_samples_split': 2, 'n_estimators': 200}


In [107]:
deployment_package = {
    'model': best_model,
    'scaler': scaler,
    'feature_names': list(x_train.columns)
}

# 2. Save the package to your local directory
joblib.dump(deployment_package, 'churn_model_package.pkl')

print("Model and preprocessing artifacts saved successfully.")

Model and preprocessing artifacts saved successfully.


In [99]:
# Save comparison table containing all new metrics for Streamlit display
comparison_df.to_csv('artifacts/model_comparison.csv', index=False)

OSError: Cannot save file into a non-existent directory: 'artifacts'